In [7]:
import pandas as pd
from pathlib import Path

def cruzar_estado(excel_path, json_path, output_path):
    # 1) Leo el Excel y normalizo sólo los dígitos en _proj_int
    df_xl = pd.read_excel(excel_path, dtype=str)
    df_xl['_proj_int'] = (
        df_xl['NumeroProyecto']
           .str.extract(r'(\d+)', expand=False)  # toma sólo números
           .fillna('0')                          # convierte NaN → '0'
           .astype(int)
    )

    # 2) Cargo el JSON y construyo el mapa
    df_j = pd.read_json(json_path, orient='records', dtype=str)
    registros = []

    for row in df_j.to_dict(orient='records'):
        # Archivados
        pa = row.get("Proyectos Archivados")
        aa = row.get("Año Archivados")
        if isinstance(pa, str) and pa.isdigit():
            num = int(pa)
            # sólo añado si el año también es un dígito válido
            year = int(aa) if isinstance(aa, str) and aa.isdigit() else None
            registros.append({
                '_proj_int': num,
                'estado': 'archivado',
                'anio_estado': year
            })

        # Retirados
        pr = row.get("Proyectos Retirados")
        ar = row.get("Año Retirados")
        if isinstance(pr, str) and pr.isdigit():
            num = int(pr)
            year = int(ar) if isinstance(ar, str) and ar.isdigit() else None
            registros.append({
                '_proj_int': num,
                'estado': 'retirado',
                'anio_estado': year
            })

    df_map = pd.DataFrame(registros)
    # Si un proyecto sale en ambas categorías, nos quedamos con 'retirado'
    df_map = df_map.drop_duplicates(subset='_proj_int', keep='last')

    # 3) Merge con el Excel
    df_out = df_xl.merge(df_map, on='_proj_int', how='left')

    # 4) Limpio columna auxiliar y guardo
    df_out = df_out.drop(columns=['_proj_int'])
    df_out.to_excel(output_path, index=False)
    print(f"✅ Generado: {Path(output_path).resolve()}")

if __name__ == '__main__':
    cruzar_estado(
        excel_path  = '2018-2019.xlsx',           # tu Excel
        json_path   = 'novedades_v1.json',        # tu JSON de proyectos
        output_path = '2018-2019_con_estado.xlsx' # salida deseada
    )


✅ Generado: C:\Users\juans\Documents\proarchitecg\Model-Extract-information\extract\modelos\team\2018-2019_con_estado.xlsx
